# 01 — Simple Graph: Mood Detector & Responder

The **"Hello World"** of LangGraph — no LLM or API key required.

### Core concepts covered
| Concept | Description |
|---------|-------------|
| `TypedDict` State | Shared data object that flows through the graph |
| Node | Plain Python function that reads + updates state |
| Sequential Edge | Fixed connection — node A always goes to node B |
| `compile()` & `invoke()` | Build and run the graph |

### Graph flow
```
START → detect_mood → craft_response → format_output → END
```

## 1. Install Dependencies

In [ ]:
# Run this cell once if langgraph is not installed
# !pip install langgraph

## 2. Imports

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

## 3. Define the State

The `State` is a `TypedDict` — a typed dictionary that every node reads from and writes to.
Only the keys you return from a node are updated; the rest stay as-is.

In [ ]:
class MoodState(TypedDict):
    text:     str   # user input text
    mood:     str   # positive | negative | neutral
    response: str   # chosen reply message
    output:   str   # final formatted string

## 4. Define the Nodes

Each node is a plain Python function:
- Takes the full state as input
- Returns a **dict** with only the keys it wants to update

In [ ]:
def detect_mood(state: MoodState) -> dict:
    """Node 1 — classify the mood of the input text using keyword matching."""
    words    = set(state["text"].lower().split())
    positive = {"happy", "great", "awesome", "excited", "good",
                "wonderful", "love", "joy", "fantastic", "amazing"}
    negative = {"sad", "bad", "terrible", "awful", "depressed",
                "angry", "hate", "frustrated", "horrible", "worst"}

    if words & positive:
        mood = "positive"
    elif words & negative:
        mood = "negative"
    else:
        mood = "neutral"

    print(f"  [Node 1] detect_mood → {mood}")
    return {"mood": mood}

In [ ]:
def craft_response(state: MoodState) -> dict:
    """Node 2 — pick the right reply for the detected mood."""
    replies = {
        "positive": "That's wonderful to hear! Keep that positive energy going! 🌟",
        "negative": "I'm sorry you're feeling that way. Things will get better soon! 💙",
        "neutral":  "Thanks for sharing. Hope the rest of your day goes smoothly! 😊",
    }
    print("  [Node 2] craft_response → done")
    return {"response": replies[state["mood"]]}

In [ ]:
def format_output(state: MoodState) -> dict:
    """Node 3 — format everything into a clean result."""
    output = (
        f"\n{'─'*45}\n"
        f"  Input : {state['text']}\n"
        f"  Mood  : {state['mood'].upper()}\n"
        f"  Reply : {state['response']}\n"
        f"{'─'*45}"
    )
    print("  [Node 3] format_output → done")
    return {"output": output}

## 5. Build & Compile the Graph

1. Create a `StateGraph` with your state type
2. `add_node()` — register each function as a node
3. `add_edge()` — connect nodes in order
4. `compile()` — lock the graph into a runnable object

In [ ]:
builder = StateGraph(MoodState)

# Register nodes
builder.add_node("detect_mood",    detect_mood)
builder.add_node("craft_response", craft_response)
builder.add_node("format_output",  format_output)

# Connect them sequentially
builder.add_edge(START,            "detect_mood")
builder.add_edge("detect_mood",    "craft_response")
builder.add_edge("craft_response", "format_output")
builder.add_edge("format_output",  END)

graph = builder.compile()
print("Graph compiled successfully!")

## 6. Visualise the Graph *(optional)*

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Visualisation requires extra deps (pygraphviz). Skipping. Error: {e}")

## 7. Run the Graph

In [ ]:
test_cases = [
    "I'm feeling so happy and excited today!",
    "This is a terrible and horrible experience.",
    "Just finished my second cup of coffee.",
]

for text in test_cases:
    print(f'\nRunning graph for: "{text}"')
    result = graph.invoke({"text": text, "mood": "", "response": "", "output": ""})
    print(result["output"])